In [ ]:
#!pip install arch

In [1]:
from arch import arch_model
import pandas as pd
import wrds
import numpy as np

/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
nasbramlidb =\
(
    wrds
    .Connection()
)

Enter your WRDS username [nasruddinislambinramli]:nasbramli
Enter your password:········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [3]:
fama_french_query =\
(
    """
    SELECT date, mktrf, smb, hml, rf 
    FROM ff.factors_monthly
    WHERE date >= '2000-01-01' AND date < '2024-01-01'
    """
)

In [4]:
ff_df =\
(
    db
    .raw_sql(fama_french_query)
)

In [5]:
(
    ff_df
    .to_csv('fama_french.csv', index = False)
)

In [6]:
fama_french =\
(
    pd
    .read_csv('fama_french.csv')
)

In [7]:
fama_french

,date,mktrf,smb,hml,rf
0,2000-01-01,-0.0474,0.0577,-0.0188,0.0041
1,2000-02-01,0.0245,0.2136,-0.0959,0.0043
2,2000-03-01,0.0520,-0.1720,0.0813,0.0047
3,2000-04-01,-0.0640,-0.0668,0.0726,0.0046
4,2000-05-01,-0.0442,-0.0605,0.0475,0.0050
...,...,...,...,...,...
283,2023-08-01,-0.0239,-0.0320,-0.0108,0.0045
284,2023-09-01,-0.0524,-0.0249,0.0145,0.0043
285,2023-10-01,-0.0318,-0.0388,0.0019,0.0047
286,2023-11-01,0.0883,-0.0003,0.0166,0.0044


In [8]:
fama_french["Date"]=\
(
    fama_french
    .date
    .apply(lambda x: str(x)
          )
)

fama_french["Date"]=\
(
    pd
    .to_datetime(fama_french.Date,
                 format="%Y-%m-%d"
                )
)

fama_french =\
(
    fama_french
    .set_index("Date", inplace = False)
)


In [9]:
fama_french =\
(
    fama_french
    .drop(columns=['date'])
)

In [10]:
fama_french

,mktrf,smb,hml,rf
Date,,,,
2000-01-01,-0.0474,0.0577,-0.0188,0.0041
2000-02-01,0.0245,0.2136,-0.0959,0.0043
2000-03-01,0.0520,-0.1720,0.0813,0.0047
2000-04-01,-0.0640,-0.0668,0.0726,0.0046
2000-05-01,-0.0442,-0.0605,0.0475,0.0050
...,...,...,...,...
2023-08-01,-0.0239,-0.0320,-0.0108,0.0045
2023-09-01,-0.0524,-0.0249,0.0145,0.0043
2023-10-01,-0.0318,-0.0388,0.0019,0.0047


In [11]:
fama_french =\
(
    fama_french[['smb', 'hml']]
)

In [12]:
fama_french

,smb,hml
Date,,
2000-01-01,0.0577,-0.0188
2000-02-01,0.2136,-0.0959
2000-03-01,-0.1720,0.0813
2000-04-01,-0.0668,0.0726
2000-05-01,-0.0605,0.0475
...,...,...
2023-08-01,-0.0320,-0.0108
2023-09-01,-0.0249,0.0145
2023-10-01,-0.0388,0.0019


In [13]:
file_path =\
(
    'all_tickers_time_series.hf5'
)

with pd.HDFStore(file_path, 'r') as hdf_file:
    keys =\
    (
        hdf_file.keys()
    )

In [14]:
russell = {}

for key in keys:
    
    data =\
    (
        pd
        .read_hdf('all_tickers_time_series.hf5', 
                  key=key)
    )
    
    if isinstance(data, pd.Series):
        
        data =\
        (
            data
            .to_frame()
        )
        
    data =\
    (
        data
        .drop_duplicates()
    )
    
    data =\
    (
        data
        .dropna()
    )
    
    if isinstance(data, pd.DataFrame) and 'date' in data.columns:
        data['Date'] =\
        (
            pd
            .to_datetime(data['date'], 
                         format="%Y-%m-%d")
        )
        
        data =\
        (
            data
            .drop(columns=['date'])
        )
        
        data =\
        (
            data
            .set_index('Date')
        )
    
    
    russell[key] = data

In [15]:
stock_data = {}

In [16]:
for ticker, data in russell.items():

    df =\
    (
        pd
        .DataFrame(data)
    )

    print(f"Columns for {ticker}: {df.columns.tolist()}")
    
    if 'prc' in df.columns:
        
        df['returns'] =\
        (
            np.log(df['prc']
                   /
                   df['prc'].shift(1)
                  )
        )
        
        df['r_vol'] =\
        (
            df['returns']
            .rolling(window = 252)
            .std()
            *
            np
            .sqrt
            (252)
        )
        
        
        stock_data[ticker] = df
    else:
        print(f"'prc' column not found for {ticker}. Available columns: {df.columns.tolist()}")


Columns for /A: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /AA: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /AAL: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /AAON: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /AAP: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /AAPL: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /ABBV: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /ABNB: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /ABT: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']


Columns for /CHRD: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /CHRW: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /CHTR: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /CI: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /CIEN: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /CINF: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /CIVI: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /CL: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /CLF: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc

Columns for /FNF: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /FOUR: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /FOX: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /FOXA: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /FR: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /FRPT: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /FRT: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /FSLR: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /FTI: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc'

Columns for /MRK: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /MRNA: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /MRO: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /MRVL: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /MS: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /MSA: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /MSCI: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /MSFT: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /MSGS: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc

Columns for /RVTY: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /RYAN: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /RYN: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /S: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /SAIA: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /SAIC: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /SAM: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /SBAC: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /SBUX: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc

Columns for /WSO: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /WST: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /WTFC: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /WTM: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /WTRG: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /WTW: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /WU: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /WWD: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
Columns for /WY: ['ticker', 'openprc', 'askhi', 'bidlo', 'prc', 'vol', 'dlstdt', 'dlstcd', 'nextdt', 'dlprc']
C

In [17]:
# Check for duplicates in returns
for ticker, data in stock_data.items():
    
    r_vol =\
    (
        data['r_vol']
    )
    
    # Check for duplicates
    if r_vol.index.duplicated().any():
        
        print(f"Duplicates found in returns for {ticker}. Resolving...")
        
        r_vol =\
        (
            r_vol[~r_vol.index.duplicated(keep='first')]  # Keep the first occurrence
        )
        
    # Check for duplicates in Fama-French factors
    if fama_french.index.duplicated().any():
        
        print("Duplicates found in Fama-French factors. Resolving...")
        
        fama_french =\
        (
            fama_french[~fama_french.index.duplicated(keep='first')]  # Keep the first occurrence
        )

    # Align with Fama-French factors
    aligned_r_vol =\
    (
        r_vol
        .reindex(fama_french.index)
        .dropna()
    )
    
    aligned_factors =\
    (
        fama_french
        .loc[aligned_r_vol.index] # Align factors with returns
    )

    # Now both aligned_returns and aligned_factors have the same index
    print(f"Aligned returns for {ticker}:\n{aligned_r_vol.head()}")
    
    print(f"Aligned Fama-French factors:\n{aligned_factors.head()}")
    
    # Fit the GARCH-X model
    model =\
    (
        arch_model(aligned_r_vol,
                   vol='Garch',
                   p=1, 
                   q=1,
                   x=aligned_factors.values,
                   dist='Normal')
    )
    
    try:
        results =\
        (
            model
            .fit(disp='off')
        )
        
        print(f"Results for {ticker}:")
        
        print(results.summary())
        
    except Exception as e:
        
        print(f"Error fitting GARCH model for {ticker}: {e}")


Aligned returns for /A:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /A: first_obs and last_obs produce in an empty array.
Aligned returns for /AA:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /AA: first_obs and last_obs produce in an empty array.
Aligned returns for /AAL:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /AAL: first_obs and last_obs produce in an empty array.
Aligned returns for /AAON:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /AAON: first_obs and last_obs produce in an empty array.
Aligned returns for /AAP:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
E

/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.009579. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.003327. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
/Users/nasruddinislambinraml

                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  r_vol   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:                17.3883
Distribution:                  Normal   AIC:                          -26.7767
Method:            Maximum Likelihood   BIC:                          -23.9445
                                        No. Observations:                   15
Date:                Sun, Oct 13 2024   Df Residuals:                       14
Time:                        22:28:01   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             0.4291  8.528e-02      5.031  4.872e-07 [  0.262,  0.59

Error fitting GARCH model for /AVY: first_obs and last_obs produce in an empty array.
Aligned returns for /AWI:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /AWI: first_obs and last_obs produce in an empty array.
Aligned returns for /AWK:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /AWK: first_obs and last_obs produce in an empty array.
Aligned returns for /AXON:
Date
2016-07-01    0.832911
2016-08-01    0.796160
2016-09-01    0.740765
2016-11-01    0.660786
2016-12-01    0.647598
Name: r_vol, dtype: float64
Aligned Fama-French factors:
               smb     hml
Date                      
2016-07-01  0.0249 -0.0132
2016-08-01  0.0116  0.0318
2016-09-01  0.0212 -0.0124
2016-11-01  0.0571  0.0821
2016-12-01  0.0010  0.0353
Results for /AXON:
                     Constant Mean - GARCH Model

/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.002277. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.02052. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
/Users/nasruddinislambinramli

...

Results for /TKO:
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  r_vol   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:                27.0106
Distribution:                  Normal   AIC:                          -46.0211
Method:            Maximum Likelihood   BIC:                          -39.6871
                                        No. Observations:                   36
Date:                Sun, Oct 13 2024   Df Residuals:                       35
Time:                        22:28:05   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             0.7103  1.418e-02     50.085      0.0

/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 7.343e-05. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.06598. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(


Error fitting GARCH model for /TRV: first_obs and last_obs produce in an empty array.
Aligned returns for /TSCO:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /TSCO: first_obs and last_obs produce in an empty array.
Aligned returns for /TSLA:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /TSLA: first_obs and last_obs produce in an empty array.
Aligned returns for /TSN:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /TSN: first_obs and last_obs produce in an empty array.
Aligned returns for /TT:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /TT: first_obs and last_obs produce in an empty array.
Alig

Error fitting GARCH model for /VTRS: first_obs and last_obs produce in an empty array.
Aligned returns for /VVV:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /VVV: first_obs and last_obs produce in an empty array.
Aligned returns for /VZ:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /VZ: first_obs and last_obs produce in an empty array.
Aligned returns for /W:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /W: first_obs and last_obs produce in an empty array.
Aligned returns for /WAB:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [smb, hml]
Index: []
Error fitting GARCH model for /WAB: first_obs and last_obs produce in an empty array.
Aligned ret

/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.07355. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.005653. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(


In [19]:
for ticker, data in stock_data.items():
    
    r_vol = data['r_vol']
    
    if r_vol.index.duplicated().any():
        print(f"Duplicates found in returns for {ticker}. Resolving...")
        r_vol =\
        (
            r_vol
            [~r_vol.index.duplicated(keep='first')]
        )
    
    if fama_french.index.duplicated().any():
        print("Duplicates found in Fama-French factors. Resolving...")
        fama_french =\
        (
            fama_french
            [~fama_french.index.duplicated(keep='first')]
        )

    r_vol.index =\
    (
        pd
        .to_datetime
        (r_vol.index)
    )
    
    fama_french.index =\
    (
        pd
        .to_datetime
        (fama_french.index)
    )
    
    aligned_factors =\
    (
        pd
        .merge_asof(r_vol.sort_index(), 
                    fama_french.sort_index(), 
                    left_index=True, 
                    right_index=True, 
                    direction='backward')
    )
    
    aligned_factors =\
    (
        aligned_factors
        .dropna()
    )

    aligned_r_vol =\
    (
        r_vol
        .loc
        [aligned_factors.index]
    )

    print(f"Aligned returns for {ticker}:\n{aligned_r_vol.head()}")
    
    print(f"Aligned Fama-French factors:\n{aligned_factors.head()}")
    
    model =\
    (
        arch_model(aligned_r_vol,
                   vol='Garch',
                   p=1, 
                   q=1,
                   x=aligned_factors.values,
                   dist='Normal')
    )
    
    try:
        results =\
        (
            model
            .fit
            (disp='off')
        )
        
        print(f"Results for {ticker}:")
        
        print(results.summary())
        
    except Exception as e:
        
        print(f"Error fitting GARCH model for {ticker}: {e}")

Aligned returns for /A:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [r_vol, smb, hml]
Index: []
Error fitting GARCH model for /A: first_obs and last_obs produce in an empty array.
Aligned returns for /AA:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [r_vol, smb, hml]
Index: []
Error fitting GARCH model for /AA: first_obs and last_obs produce in an empty array.
Aligned returns for /AAL:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [r_vol, smb, hml]
Index: []
Error fitting GARCH model for /AAL: first_obs and last_obs produce in an empty array.
Aligned returns for /AAON:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [r_vol, smb, hml]
Index: []
Error fitting GARCH model for /AAON: first_obs and last_obs produce in an empty array.
Aligned returns for /AAP:
Series([], Name: r_vol, dtype: float64)
Al

/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.01805. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.07429. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(


Error fitting GARCH model for /ALGM: first_obs and last_obs produce in an empty array.
Aligned returns for /ALGN:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [r_vol, smb, hml]
Index: []
Error fitting GARCH model for /ALGN: first_obs and last_obs produce in an empty array.
Aligned returns for /ALK:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [r_vol, smb, hml]
Index: []
Error fitting GARCH model for /ALK: first_obs and last_obs produce in an empty array.
Aligned returns for /ALL:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [r_vol, smb, hml]
Index: []
Error fitting GARCH model for /ALL: first_obs and last_obs produce in an empty array.
Aligned returns for /ALLE:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [r_vol, smb, hml]
Index: []
Error fitting GARCH model for /ALLE: first_obs and last_obs 

/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.04169. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(


Results for /APG:
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  r_vol   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:                463.136
Distribution:                  Normal   AIC:                          -918.273
Method:            Maximum Likelihood   BIC:                          -903.580
                                        No. Observations:                  291
Date:                Mon, Oct 14 2024   Df Residuals:                      290
Time:                        14:49:24   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             1.5897  7.644e-04   2079.553      0.0

...

/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.06715. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(


Results for /U:
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  r_vol   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:                289.354
Distribution:                  Normal   AIC:                          -570.708
Method:            Maximum Likelihood   BIC:                          -554.732
                                        No. Observations:                  401
Date:                Mon, Oct 14 2024   Df Residuals:                      400
Time:                        14:49:33   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             1.0763  5.428e-04   1982.734      0.000

Error fitting GARCH model for /WEC: first_obs and last_obs produce in an empty array.
Aligned returns for /WEN:
Series([], Name: r_vol, dtype: float64)
Aligned Fama-French factors:
Empty DataFrame
Columns: [r_vol, smb, hml]
Index: []
Error fitting GARCH model for /WEN: first_obs and last_obs produce in an empty array.
Aligned returns for /WEX:
Date
2001-02-05    0.905441
2001-02-08    0.904978
2001-02-12    0.905666
2001-02-13    0.924575
2001-02-14    0.922923
Name: r_vol, dtype: float64
Aligned Fama-French factors:
               r_vol     smb     hml
Date                                
2001-02-05  0.905441 -0.0078  0.1247
2001-02-08  0.904978 -0.0078  0.1247
2001-02-12  0.905666 -0.0078  0.1247
2001-02-13  0.924575 -0.0078  0.1247
2001-02-14  0.922923 -0.0078  0.1247
Results for /WEX:
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  r_vol   R-squared:                       0.000
Mean Model:             Constant Mean   Ad

/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.07631. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
/Users/nasruddinislambinramli/anaconda3/lib/python3.11/site-packages/arch/univariate/base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.01861. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 10 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(


Results for /WLK:
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                  r_vol   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:                104.287
Distribution:                  Normal   AIC:                          -200.574
Method:            Maximum Likelihood   BIC:                          -191.251
                                        No. Observations:                   76
Date:                Mon, Oct 14 2024   Df Residuals:                       75
Time:                        14:49:34   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             1.5712  8.564e-04   1834.589      0.0